In [ ]:
#importing prerequisites
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import calendar

In [ ]:

# To get the weather or meteo data
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 50.08,
    "longitude": 7.76,
    "start_date": "2000-01-01",
    "end_date": "2026-09-01",
    "hourly": (
        "temperature_2m,"
        "relative_humidity_2m,"
        "precipitation,"
        "rain,"
        "snowfall,"
        "pressure_msl,"
        "wind_speed_10m,"
        "wind_direction_10m,"
        "et0_fao_evapotranspiration"
    ),
    "timezone": "Europe/Berlin"
}

response = requests.get(url, params=params)

data = response.json()

weather = pd.DataFrame(data["hourly"])

weather["time"] = pd.to_datetime(weather["time"])

print(weather.head())
print(weather.shape)

In [ ]:
weather.head()

In [ ]:
weather.shape

In [ ]:
# Read CSV file for water level for every 15 min
df_water15 = pd.read_csv("../data/raw/water_level_15min_kaub.csv")

df_water15.head()

In [ ]:
# Read without semicolon
df_water15 = pd.read_csv("../data/raw/water_level_15min_kaub.csv",sep=";")

df_water15.head()

In [ ]:
# Converting timestamp to datetime
df_water15["timestamp"] = pd.to_datetime(df_water15["timestamp"])

# 15-minute  to hourly mean
df_water_hourly = (
    df_water15
    .set_index("timestamp")
    .resample("1h")["value"]
    .mean()
    .reset_index()
)

df_water_hourly.head()

In [ ]:
# Make sure timestamp is datetime
df_water_hourly["timestamp"] = pd.to_datetime(df_water_hourly["timestamp"])

# Create separate date and time columns
df_water_hourly["date"] = df_water_hourly["timestamp"].dt.date
df_water_hourly["time"] = df_water_hourly["timestamp"].dt.time

# Keep the required columns
df_hourly_water = df_water_hourly[["date", "time", "value"]]

df_hourly_water.head()

In [ ]:
# shape of hourly water level data
df_hourly_water.shape

In [ ]:
weather.head()

In [ ]:
# Convert to datetime
weather["time"] = pd.to_datetime(weather["time"])

# Create separate date and time columns
weather["date"] = weather["time"].dt.date
weather["time_only"] = weather["time"].dt.time

# Remove original datetime column
weather = weather.drop(columns=["time"])

# Rename time_only to time
weather = weather.rename(columns={"time_only": "time"})

In [ ]:
weather = weather[
    ["date", "time"] +
    [col for col in weather.columns if col not in ["date", "time"]]
]

weather.head()

In [ ]:
#drop first row due to time mismatch with water level data
weather = weather.iloc[1:].reset_index(drop=True)

weather.head()

In [ ]:
# Save processed weather data
#weather.to_csv("../data/processed/kaub_weather_hourly.csv",index=False)

In [ ]:
#water level hourly data
df_hourly_water.head(20)

In [ ]:
# Save hourly water level data to processed folder
df_hourly_water.to_csv("../data/processed/kaub_water_level_hourly.csv",index=False)

In [ ]:
#Checking for the NA's and duplicates
print(df_hourly_water.shape)
print(weather.shape)

print(df_hourly_water.isna().sum())
print(weather.isna().sum())

print(df_hourly_water.duplicated().sum())
print(weather.duplicated().sum())

In [ ]:
# Checking for the date and time of both the dataset
print(df_hourly_water[["date", "time"]].head(10))
print(weather[["date", "time"]].head(10))

In [ ]:
#checking for the date and time of the whole dataset
same_date = df_hourly_water["date"].reset_index(drop=True).equals(
    weather["date"].reset_index(drop=True)
)

same_time = df_hourly_water["time"].reset_index(drop=True).equals(
    weather["time"].reset_index(drop=True)
)

print("Dates are same:", same_date)
print("Times are same:", same_time)

In [ ]:
# One Final check for the missing data
water_times = set(zip(df_hourly_water["date"], df_hourly_water["time"]))
weather_times = set(zip(weather["date"], weather["time"]))

print("Missing in weather:", len(water_times - weather_times))
print("Missing in water:", len(weather_times - water_times))

In [ ]:
#Merge both data set
merged_df = pd.merge(
   df_hourly_water,
    weather,
    on=["date", "time"],
    how="inner"
)

merged_df.head()

In [ ]:
print("water level:", df_hourly_water.shape)
print("Weather:", weather.shape)
print("Merged:", merged_df.shape)

In [ ]:
merged_df.to_csv("../data/processed/kaub_water_weather_hourly.csv",index=False)

In [ ]:

df = pd.read_csv(
    "../data/processed/kaub_water_weather_hourly.csv"
)

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df["value"].describe()

In [ ]:

plt.figure(figsize=(16, 6))

plt.plot(
    df["date"],
    df["value"],
    linewidth=0.7
)

plt.title("Kaub Water Level Over Time")
plt.xlabel("Date")
plt.ylabel("Water Level (cm)")

plt.tight_layout()
plt.show()

In [ ]:
df["datetime"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["time"].astype(str)
)

In [ ]:
print("Start:", df["datetime"].min())
print("End:", df["datetime"].max())

In [ ]:
#Kaub water level over the time

plt.figure(figsize=(16, 6))

plt.plot(
    df["datetime"],
    df["value"],   # use your actual water-level column
    linewidth=0.7
)

plt.title("Kaub Water Level Over Time")
plt.xlabel("Year")
plt.ylabel("Water Level (cm)")

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df["month"] = df["datetime"].dt.month

In [ ]:
#Monthly avaerage 
monthly_avg = (
    df.groupby("month")["value"]
      .mean()
)

In [ ]:
#Kaub water level by month
plt.figure(figsize=(10, 5))
monthly_avg.plot(kind="bar")

plt.title("Kaub Water Level by Month")
plt.xlabel("Month")
plt.ylabel("Average Water Level (cm)")
plt.xticks(range(len(monthly_avg)), [calendar.month_abbr[m] for m in monthly_avg.index], rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Make sure datetime exists
df["datetime"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["time"].astype(str)
)

# Daily average from hourly water-level data
daily_water = (
    df.set_index("datetime")["value"]
      .resample("D")
      .mean()
)

# 365-day rolling mean to visualize long-term trend
trend = daily_water.rolling(
    window=365,
    center=True
).mean()

In [ ]:
#To see the long time trend
plt.figure(figsize=(16, 6))

# Daily fluctuations
plt.plot(
    daily_water.index,
    daily_water,
    alpha=0.3,
    linewidth=0.7,
    label="Daily Average"
)

# Long-term trend
plt.plot(
    trend.index,
    trend,
    linewidth=2,
    label="365-Day Rolling Average"
)

plt.title("Kaub Water Level: Long-Term Trend")
plt.xlabel("Year")
plt.ylabel("Water Level (cm)")

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
#To see the seasonal pattern
df["month"] = df["datetime"].dt.month
df["year"] = df["datetime"].dt.year

monthly_avg = (
    df.groupby("month")["value"]
      .mean()
)

monthly_avg

month_names = [
    "Jan", "Feb", "Mar", "Apr",
    "May", "Jun", "Jul", "Aug",
    "Sep", "Oct", "Nov", "Dec"
]

plt.figure(figsize=(11, 5))

plt.plot(
    monthly_avg.index,
    monthly_avg.values,
    marker="o"
)

plt.xticks(range(1, 13), month_names)

plt.title("Seasonal Pattern of Kaub Water Level")
plt.xlabel("Month")
plt.ylabel("Average Water Level (cm)")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Make sure date is datetime
merged_df["date"] = pd.to_datetime(merged_df["date"])

# Convert hourly data to daily data
daily_df = merged_df.groupby("date").agg({
    "value": ["mean", "min", "max"],
    "temperature_2m": "mean",
    "relative_humidity_2m": "mean",
    "precipitation": "sum",
    "rain": "sum",
    "snowfall": "sum",
    "pressure_msl": "mean",
    "wind_speed_10m": "mean"
}).reset_index()

In [ ]:

daily_df.columns = [
    "date",
    "water_level_mean",
    "water_level_min",
    "water_level_max",
    "temperature_mean",
    "humidity_mean",
    "precipitation_sum",
    "rain_sum",
    "snowfall_sum",
    "pressure_mean",
    "wind_speed_mean"
]

In [ ]:
daily_df.head()

In [ ]:
daily_df.shape

In [ ]:
# Check date range
print("Start:", daily_df["date"].min())
print("End:", daily_df["date"].max())

# Missing values
print(daily_df.isnull().sum())

# Duplicated dates
print("Duplicates:", daily_df["date"].duplicated().sum())

# Basic statistics
daily_df.describe()

In [ ]:
#To find the missing days
expected_dates = pd.date_range(
    daily_df["date"].min(),
    daily_df["date"].max(),
    freq="D"
)

missing_dates = expected_dates.difference(daily_df["date"])

print("Number of missing days:", len(missing_dates))

In [ ]:
#information on data
daily_df.info()

In [ ]:
#To see the summary
daily_df.describe()

In [ ]:
#To check the null values
daily_df.isnull().sum()

In [ ]:
#Summary of the data
daily_df["water_level_mean"].describe()

In [ ]:
daily_df.to_csv("../data/processed/kaub_water_weather_daily.csv",index=False)

In [ ]:

#Daily water level
plt.figure(figsize=(15, 5))
plt.plot(daily_df["date"], daily_df["water_level_mean"])
plt.xlabel("Date")
plt.ylabel("Water Level (cm)")
plt.title("Daily Water Level at Kaub")
plt.show()

In [ ]:
daily_df["water_level_mean"].describe()

In [ ]:
daily_df.nsmallest(20, "water_level_mean")[
    ["date", "water_level_mean", "water_level_min", "water_level_max"]
]

In [ ]:
# to check the complete 24 hours
hours_per_day = (
    merged_df
    .groupby("date")
    .size()
    .reset_index(name="hours_available")
)

hours_per_day["hours_available"].value_counts().sort_index()

In [ ]:
#merge the water level on date
daily_check = daily_df.merge(
    hours_per_day,
    on="date",
    how="left"
)

In [ ]:
#To check the 20 minimum water level
daily_check.nsmallest(20, "water_level_mean")[
    [
        "date",
        "water_level_mean",
        "water_level_min",
        "water_level_max",
        "hours_available"
    ]
]

In [ ]:
#To check the daily water level pattern

plt.figure(figsize=(16, 6))

plt.plot(
    daily_df["date"],
    daily_df["water_level_mean"]
)

plt.xlabel("Date")
plt.ylabel("Water Level (cm)")
plt.title("Daily Mean Water Level at Kaub")

plt.grid(alpha=0.3)
plt.show()